In [40]:
#######################################################################################
## Model 2 : Complex - Exact DP ( ☆☆☆ will take a very long time) ##
#######################################################################################

import numpy as np
from scipy.stats import norm
from numpy.polynomial.hermite import hermgauss

# Parameter
AGE_MIN, AGE_MAX = 18, 25
T = AGE_MAX - AGE_MIN

delta = 0.95
rho = 0.95
tuition = 3000

# wage = base_wage + shock
wage_base = lambda educ, skill: 5000 + 5000 * educ + 3000 * skill  # base wage
sigma_eps = 1500  # standard deviation of wage shock

# State： (age, education, asset, skill, exp, parent_income, ability)
actions = ['study', 'work', 'delay']
asset_grid = np.round(np.linspace(-20000, 50000, 51), 2)
skill_grid = np.round(np.linspace(0.0, 1.0, 21), 2)
exp_grid = list(range(0, 8))
parent_income_grid = [0, 1]  # 0 = low-income, 1 = high-income
educ_levels = [0, 1, 2, 3, 4]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}
ability_transition = {
    'low':    {'low': 0.5, 'medium': 0.4, 'high': 0.1},
    'medium': {'low': 0.1, 'medium': 0.6, 'high': 0.3},
    'high':   {'low': 0.0, 'medium': 0.2, 'high': 0.8}
}

def closest_grid_value(x, grid):
    return float(grid[np.argmin(np.abs(grid - x))])

# Initialize the state space
states = []
V = {}
policy = {}
state_set = set()
for age in range(AGE_MIN, AGE_MAX+1):
    for educ in educ_levels:
        for asset in asset_grid:
            for skill in skill_grid:
                for exp in exp_grid:
                    for p_inc in parent_income_grid:
                        for ability in abilities:
                            s = (age, educ, asset, skill, exp, p_inc, ability)
                            states.append(s)
                            V[s] = 0.0
                            policy[s] = None
                            state_set.add(s)


def crra(c, rho):
    return np.sign(c) * (abs(c) ** (1 - rho)) / (1 - rho)

def behavior_utility(action):
    return {'study': -0.05, 'work': -10, 'delay': -2}[action]

def terminal_reward(educ, asset):
    return crra(max(asset, 1.0), rho) + 0.5 * educ

prob_success_matrix = np.array([
    [0.7, 0.7, 0.6, 0.5, 0.5],  # low
    [0.8, 0.7, 0.6, 0.6, 0.6],  # medium
    [0.9, 0.8, 0.7, 0.7, 0.7]   # high
])

def prob_educ_success(ability, educ):
    return prob_success_matrix[ability_map[ability]][educ]

def expected_value_gauss_hermite(f, mu=0, sigma=1, n=5):
    x, w = hermgauss(n)
    x = np.sqrt(2) * sigma * x + mu
    w = w / np.sqrt(np.pi)
    return sum(w[i] * f(x[i]) for i in range(n))

# Terminal reward
for s in states:
    age, educ, asset, skill, exp, p_inc, ability = s
    if age == AGE_MAX:
        V[s] = terminal_reward(educ, asset)

# Backward value iteration
for t in reversed(range(AGE_MIN, AGE_MAX)):
    for s in states:
        age, educ, asset, skill, exp, p_inc, ability = s
        if age != t:
            continue

        best_v = -np.inf
        best_a = None

        for a_type in actions:
            for asset_next in asset_grid:
                if asset_next < -20000:
                    continue

                def integrand(eps):
                    # skill involvement
                    skill_next = min(1.0, skill + (0.05 if a_type == 'study' else 0.02))
                    skill_next = closest_grid_value(skill_next, skill_grid)
                    # exp involvement
                    exp_next = min(7, exp + 1) if a_type == 'work' else exp

                    # wage, cost, consumption
                    wage = 0
                    cost = 0
                    if a_type == 'work':
                        wage = wage_base(educ, skill) + eps
                    elif a_type == 'study':
                        cost = tuition - (1000 if p_inc == 0 else 0)

                    consumption = wage - cost - (asset_next - asset)
                    if consumption <= 0:
                       return -1e10
                    u = crra(consumption, rho) + behavior_utility(a_type)

                    # educ level depends on prob_educ_success
                    p_success = prob_educ_success(ability, educ)
                    next_educ = min(4, educ + 1)  # if success then +1

                    # ability changes with markov transition matrix ✅
                    value_total = 0.0
                    for ability_next, p_ability in ability_transition[ability].items():
                        # 2 possibilities of futures
                        s1 = (age + 1, next_educ, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)
                        s2 = (age + 1, educ, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)
                        v1 = V.get(s1, -1e10)
                        v2 = V.get(s2, -1e10)
                        value_total += p_ability * (p_success * v1 + (1 - p_success) * v2)

                    return u + delta * value_total
                # Gausse-Hermite quadrature to calculate the expectation
                val = expected_value_gauss_hermite(integrand, mu=0, sigma=sigma_eps, n=5)
                if val > best_v:
                    best_v = val
                    best_a = (a_type, closest_grid_value(asset_next, asset_grid))

        V[s] = best_v
        policy[s] = best_a

# Show the policy trajectory
print("\n✅ Exact DP completed.\n")

print("Exact policy trajectory (CRRA, extended state) from state (18, 0, 0.0, 0.0, 0, 1, 'high'):")
state = (18, 0, closest_grid_value(0.0, asset_grid), closest_grid_value(0.0, skill_grid), 0, 1, 'high')
for age in range(AGE_MIN, AGE_MAX):
    action = policy.get(state, None)
    value = V.get(state, None)
    if action is None or value is None or state not in state_set:
        print(f"Age {age}: No valid action or value found for state {state}")
        break

    print(f"Age {age}: State = {state}, V* = {value:.3f}, Optimal Action = {action}")

    # next state
    next_educ = min(4, state[1] + 1) if action[0] == 'study' else state[1]
    next_asset = closest_grid_value(action[1], asset_grid)
    next_skill = closest_grid_value(min(1.0, state[3] + (0.05 if action[0] == 'study' else 0.02)), skill_grid)
    next_exp = min(7, state[4] + 1) if action[0] == 'work' else state[4]
    next_p_inc = state[5]

    if action[0] == 'study':
        success = True
    else:
        success = False
    ability_transit = ability_transition[state[6]]
    next_ability = max(ability_transit.items(), key=lambda x: x[1])[0]

    state = (age + 1, next_educ if success else state[1], next_asset, next_skill, next_exp, next_p_inc, next_ability)
    if state not in state_set:
        print(f"  ⚠️ Next state {state} not in known state space.")
        break

KeyboardInterrupt: 

In [41]:
#########################################################
## Model 2 : Complex - ADP - Policy Gradient ##
#########################################################

import numpy as np
import random
from scipy.stats import norm
from numpy.polynomial.hermite import hermgauss

# Parameter
AGE_MIN, AGE_MAX = 18, 25
delta = 0.95
rho = 0.95
tuition = 3000
sigma_eps = 1500

action_types = ['study', 'work', 'delay']
ACTIONS = [(act, asset_next) for act in action_types for asset_next in np.round(np.linspace(-20000, 50000, 50), 2)]

asset_grid = np.round(np.linspace(-20000, 50000, 51), 2)
skill_grid = np.round(np.linspace(0.0, 1.0, 21), 2)
exp_grid = list(range(8))
parent_income_grid = [0, 1]
educ_levels = [0, 1, 2, 3, 4]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}

def wage_base(educ, skill):
    return 5000 + 5000 * educ + 3000 * skill

ability_transition = {
    'low':    {'low': 0.5, 'medium': 0.4, 'high': 0.1},
    'medium': {'low': 0.1, 'medium': 0.6, 'high': 0.3},
    'high':   {'low': 0.0, 'medium': 0.2, 'high': 0.8}
}

def prob_educ_success(ability, educ):
    p = np.array([
        [0.7, 0.7, 0.6, 0.5, 0.5],
        [0.8, 0.7, 0.6, 0.6, 0.6],
        [0.9, 0.8, 0.7, 0.7, 0.7]
    ])
    return p[ability_map[ability]][educ]

def crra(c, rho):
    return np.sign(c) * (abs(c)**(1 - rho)) / (1 - rho)

def behavior_utility(action):
    return {'study': -5, 'work': -10, 'delay': -2}[action]

def terminal_reward(educ, asset):
    return crra(max(asset, 1.0), rho) + 0.5 * educ

def closest_grid_value(x, grid):
    return float(grid[np.argmin(np.abs(grid - x))])

# Feature Vector
def feature_vector(state, action):
    age, educ, asset, skill, exp, p_inc, ability = state
    act, asset_next = action
    ability_idx = ability_map[ability]
    return np.array([
        age - 18, educ/4, (asset-(-20000))/50000-(-20000), skill, exp/7, p_inc, ability_idx/2,
        (asset_next-(-20000))/50000-(-20000), action_types.index(act)/2
    ], dtype=float)

# Softmax Policy
def softmax_policy(state, theta, actions=ACTIONS):
    logits = np.array([np.dot(theta, feature_vector(state, a)) for a in actions])
    exps = np.exp(logits - np.max(logits))
    return exps / np.sum(exps)

# Legal Actions
def legal_actions(state):
    age, educ, asset_now, skill, exp, p_inc, ability = state
    legal = []
    for action in ACTIONS:
        act, asset_next = action
        if asset_next < -20000:
            continue
        base_income = wage_base(educ, skill) if act == 'work' else 0
        cost = tuition - (1000 if p_inc == 0 else 0) if act == 'study' else 0
        worst_shock = -3 * sigma_eps
        income = base_income + worst_shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            continue
        legal.append(action)
    return legal

# Choose action
def choose_action(state, theta):

    legal = legal_actions(state)
    if not legal:
        None

    probs = softmax_policy(state, theta, actions=legal)
    return legal[np.random.choice(len(legal), p=probs)]

# Trajectory Simulation
def simulate_trajectory(s0, theta):
    trajectory = []
    state = s0
    for t in range(AGE_MIN, AGE_MAX):
        action = choose_action(state, theta)
        if action is None:
            break
        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action

        if asset_next < -20000:
            break

        base_income = 0
        cost = 0
        if act == 'work':
            base_income = wage_base(educ, skill)
        elif act == 'study':
            cost = tuition - (1000 if p_inc == 0 else 0)

        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            break

        reward = crra(consumption, rho) + behavior_utility(act)

        # state involvement
        skill_next = min(1.0, skill + (0.05 if act == 'study' else 0.02))
        skill_next = closest_grid_value(skill_next, skill_grid)
        exp_next = min(7, exp + 1) if act == 'work' else exp

        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            educ_next = educ + 1 if random.random() < p_succ else educ
        else:
            educ_next = educ
        educ_next = min(educ_next, 4)

        ability_next = random.choices(
            list(ability_transition[ability].keys()),
            weights=list(ability_transition[ability].values())
        )[0]

        next_state = (age + 1, educ_next, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)

        trajectory.append((state, action, reward))
        state = next_state

    final_rew = terminal_reward(state[1], state[2])
    trajectory.append((state, None, final_rew))
    return trajectory

# Compute Returns -----
def compute_returns(traj, gamma=delta):
    G = 0
    returns = []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

# Compute Policy Gradient -----
def compute_policy_gradient(traj, returns, theta):
    grads = np.zeros_like(theta)
    for (s, a, _), G in zip(traj, returns):
        if a is None:
            continue
        probs = softmax_policy(s, theta)
        phi = np.array([feature_vector(s, act) for act in ACTIONS])
        grad_logpi = feature_vector(s, a) - np.dot(probs, phi)
        grads += G * grad_logpi
    return grads

# Train the Policy Gradient -----
def train_policy_gradient(theta_init, alpha=0.01, epochs=200, episodes_per_epoch=50):
    theta = theta_init.copy()
    history = []
    for epoch in range(epochs):
        grads_all = []
        returns_all = []
        for _ in range(episodes_per_epoch):
            s0 = (18, 0, closest_grid_value(0.0, asset_grid), closest_grid_value(0.0, skill_grid), 0, 1, 'high')
            traj = simulate_trajectory(s0, theta)
            R = compute_returns(traj)
            grad = compute_policy_gradient(traj, R, theta)
            grads_all.append(grad)
            returns_all.append(R[0])
        theta += alpha * np.mean(grads_all, axis=0)
        avg_return = np.mean(returns_all)
        history.append(avg_return)
        if epoch % 10 == 0:
            print(f"[Epoch {epoch}] Average Return = {avg_return:.2f}")
    return theta, history

# Print the policy trajectory
def print_policy_trajectory(theta, start_state):
    print("\n🧭 Policy Trajectory from state", start_state)
    state = start_state
    for age in range(AGE_MIN, AGE_MAX):
        action = choose_action(state, theta)
        if action is None:
            print(f"Age {age}: State = {state}, ❌ No legal action available, terminate.")
            break

        print(f"Age {age}: State = {state}, ➡️ Chosen Action = {action}")

        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action

        base_income = wage_base(educ, skill) if act == 'work' else 0
        cost = tuition - (1000 if p_inc == 0 else 0) if act == 'study' else 0
        income = base_income if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            print(f"  ⚠️ Consumption <= 0 (without shock), terminate trajectory.")
            break

        skill_next = min(1.0, skill + (0.05 if act == 'study' else 0.02))
        skill_next = closest_grid_value(skill_next, skill_grid)
        exp_next = min(7, exp + 1) if act == 'work' else exp

        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            educ_next = educ + 1 if p_succ >= 0.5 else educ
        else:
            educ_next = educ
        educ_next = min(educ_next, 4)

        ability_next = max(ability_transition[ability].items(), key=lambda x: x[1])[0]

        state = (age + 1,
                 educ_next,
                 closest_grid_value(asset_next, asset_grid),
                 skill_next,
                 exp_next,
                 p_inc,
                 ability_next)

np.random.seed(42)
theta0 = np.random.randn(9)
theta_trained, return_history = train_policy_gradient(theta0)

# Estimate the expected utility
s0 = (18, 0, 0.0, 0.0, 0, 1, 'high')
vals = [compute_returns(simulate_trajectory(s0, theta_trained))[0] for _ in range(200)]
print(f"Estimated V(s0) ≈ {np.mean(vals):.2f}")

print_policy_trajectory(theta_trained, start_state=s0)


[Epoch 0] Average Return = 153.94
[Epoch 10] Average Return = 150.52
[Epoch 20] Average Return = 150.27
[Epoch 30] Average Return = 150.12
[Epoch 40] Average Return = 149.71
[Epoch 50] Average Return = 149.45
[Epoch 60] Average Return = 148.24
[Epoch 70] Average Return = 147.33
[Epoch 80] Average Return = 148.60
[Epoch 90] Average Return = 149.25
[Epoch 100] Average Return = 147.04
[Epoch 110] Average Return = 149.23
[Epoch 120] Average Return = 149.91
[Epoch 130] Average Return = 149.59
[Epoch 140] Average Return = 148.53
[Epoch 150] Average Return = 148.93
[Epoch 160] Average Return = 148.90
[Epoch 170] Average Return = 148.36
[Epoch 180] Average Return = 147.78
[Epoch 190] Average Return = 146.99
Estimated V(s0) ≈ 148.42

🧭 Policy Trajectory from state (18, 0, 0.0, 0.0, 0, 1, 'high')
Age 18: State = (18, 0, 0.0, 0.0, 0, 1, 'high'), ➡️ Chosen Action = ('study', np.float64(-20000.0))
Age 19: State = (19, 1, -20000.0, 0.05, 0, 1, 'high'), ➡️ Chosen Action = ('work', np.float64(-20000.0

In [42]:
####################################################
## Model 2 : Complex - ADP - Linear VFA ##
####################################################

import numpy as np
import random

# Parameter
AGE_MIN, AGE_MAX = 18, 25
delta = 0.95
rho = 0.95
tuition = 3000
sigma_eps = 1500

action_types = ['study', 'work', 'delay']
ACTIONS = [(act, asset_next) for act in action_types for asset_next in np.round(np.linspace(-20000, 50000, 50), 2)]

asset_grid = np.round(np.linspace(-20000, 50000, 51), 2)
skill_grid = np.round(np.linspace(0.0, 1.0, 21), 2)
exp_grid = list(range(8))
parent_income_grid = [0, 1]
educ_levels = [0, 1, 2, 3, 4]
abilities = ['low', 'medium', 'high']
ability_map = {'low': 0, 'medium': 1, 'high': 2}

def wage_base(educ, skill):
    return 5000 + 5000 * educ + 3000 * skill

ability_transition = {
    'low':    {'low': 0.5, 'medium': 0.4, 'high': 0.1},
    'medium': {'low': 0.1, 'medium': 0.6, 'high': 0.3},
    'high':   {'low': 0.0, 'medium': 0.2, 'high': 0.8}
}

def prob_educ_success(ability, educ):
    p = np.array([
        [0.7, 0.7, 0.6, 0.5, 0.5],
        [0.8, 0.7, 0.6, 0.6, 0.6],
        [0.9, 0.8, 0.7, 0.7, 0.7]
    ])
    return p[ability_map[ability]][educ]

def crra(c, rho):
    return np.sign(c) * (abs(c)**(1 - rho)) / (1 - rho)

def behavior_utility(action):
    return {'study': -5, 'work': -10, 'delay': -2}[action]

def terminal_reward(educ, asset):
    return crra(max(asset, 1.0), rho) + 0.5 * educ

def closest_grid_value(x, grid):
    return float(grid[np.argmin(np.abs(grid - x))])

def state_feature_vector(state):
    age, educ, asset, skill, exp, p_inc, ability = state
    ability_idx = ability_map[ability]
    return np.array([
        age - 18, educ/4, asset / 50000, skill, exp/7, p_inc, ability_idx/2
    ], dtype=float)

def value_function(state, theta):
    return np.dot(theta, state_feature_vector(state))

# Defines the action range corresponding to the approximation value. The scaling of the variable is almost the same as the Policy gradient.
def legal_actions(state):
    age, educ, asset_now, skill, exp, p_inc, ability = state
    legal = []
    for action in ACTIONS:
        act, asset_next = action
        if asset_next < -20000:
            continue
        base_income = wage_base(educ, skill) if act == 'work' else 0
        cost = tuition - (1000 if p_inc == 0 else 0) if act == 'study' else 0
        worst_shock = -3 * sigma_eps
        income = base_income + worst_shock if act == 'work' else 0
        consumption = income - cost - (asset_next - asset_now)
        if consumption <= 0:
            continue
        legal.append(action)
    return legal

def best_action(state, theta):
    legal = legal_actions(state)
    if not legal:
        return None
    best_a = None
    best_value = -np.inf
    for action in legal:
        act, asset_next = action
        age, educ, asset_now, skill, exp, p_inc, ability = state
        base_income = wage_base(educ, skill) if act == 'work' else 0
        cost = tuition - (1000 if p_inc == 0 else 0) if act == 'study' else 0
        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        reward = crra(consumption, rho) + behavior_utility(act)
        skill_next = min(1.0, skill + (0.05 if act == 'study' else 0.02))
        skill_next = closest_grid_value(skill_next, skill_grid)
        exp_next = min(7, exp + 1) if act == 'work' else exp
        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            educ_next = educ + 1 if random.random() < p_succ else educ
        else:
            educ_next = educ
        educ_next = min(educ_next, 4)
        ability_next = random.choices(['low', 'medium', 'high'],weights=[0.1, 0.6, 0.3] if ability == 'medium' else ([0.5, 0.4, 0.1] if ability == 'low' else [0.0, 0.2, 0.8]))[0]
        next_state = (age + 1, educ_next, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)
        if next_state[0] >= AGE_MAX:
            future_value = terminal_reward(next_state[1], next_state[2])
        else:
            future_value = value_function(next_state, theta)
        value = reward + delta * future_value
        if value > best_value:
            best_value = value
            best_a = action
    return best_a

# Trajectory Simulation
def simulate_trajectory(s0, theta):
    trajectory = []
    state = s0
    for age in range(AGE_MIN, AGE_MAX):
        action = best_action(state, theta)
        if action is None:
            break
        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action
        base_income = wage_base(educ, skill) if act == 'work' else 0
        cost = tuition - (1000 if p_inc == 0 else 0) if act == 'study' else 0
        shock = np.random.normal(0, sigma_eps) if act == 'work' else 0
        income = base_income + shock if act == 'work' else base_income
        consumption = income - cost - (asset_next - asset_now)
        reward = crra(consumption, rho) + behavior_utility(act)
        skill_next = min(1.0, skill + (0.05 if act == 'study' else 0.02))
        skill_next = closest_grid_value(skill_next, skill_grid)
        exp_next = min(7, exp + 1) if act == 'work' else exp
        if act == 'study':
            p_succ = prob_educ_success(ability, educ)
            educ_next = educ + 1 if random.random() < p_succ else educ
        else:
            educ_next = educ
        educ_next = min(educ_next, 4)
        ability_next = random.choices(['low', 'medium', 'high'],weights=[0.1, 0.6, 0.3] if ability == 'medium' else ([0.5, 0.4, 0.1] if ability == 'low' else [0.0, 0.2, 0.8]))[0]
        next_state = (age + 1, educ_next, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)
        trajectory.append((state, reward, next_state))
        state = next_state
        if state[0] >= AGE_MAX:
            break
    return trajectory

# Training
def train_value_iteration(theta_init, alpha=0.01, epochs=200, episodes_per_epoch=50):
    theta = theta_init.copy()
    history = []
    for epoch in range(epochs):
        all_losses = []
        for _ in range(episodes_per_epoch):
            s0 = (18, 0, closest_grid_value(0.0, asset_grid), closest_grid_value(0.0, skill_grid), 0, 1, 'high')
            traj = simulate_trajectory(s0, theta)
            for (s, r, s_next) in traj:
                target = r
                if s_next[0] >= AGE_MAX:
                    target += 0
                else:
                    target += delta * value_function(s_next, theta)
                prediction = value_function(s, theta)
                loss = 0.5 * (target - prediction)**2
                grad = (target - prediction) * state_feature_vector(s)
                theta += alpha * grad
                all_losses.append(loss)
        avg_loss = np.mean(all_losses)
        history.append(avg_loss)
        if epoch % 10 == 0:
            print(f"[Epoch {epoch}] Average Loss = {avg_loss:.6f}")
    return theta, history

# Print the greedy policy trajectory, since this is a LFM method
def print_greedy_trajectory(theta, start_state):
    print("\n🧭 Greedy Policy Trajectory from state", start_state)
    state = start_state
    for age in range(AGE_MIN, AGE_MAX):
        action = best_action(state, theta)
        if action is None:
            print(f"Age {state[0]}: {state} → ❌ No Action Available")
            break
        print(f"Age {state[0]}: State = {state}, Action = {action}")
        age, educ, asset_now, skill, exp, p_inc, ability = state
        act, asset_next = action
        skill_next = min(1.0, skill + (0.05 if act == 'study' else 0.02))
        skill_next = closest_grid_value(skill_next, skill_grid)
        exp_next = min(7, exp + 1) if act == 'work' else exp
        next_educ = educ + 1 if act == 'study' else educ
        next_educ = min(next_educ, 4)
        ability_next = max(ability_transition[ability], key=ability_transition[ability].get)
        state = (age + 1, next_educ, closest_grid_value(asset_next, asset_grid), skill_next, exp_next, p_inc, ability_next)

np.random.seed(42)
theta0 = np.random.randn(7)
theta_trained, loss_history = train_value_iteration(theta0)

# Estimate the expected value
s0 = (18, 0, 0.0, 0.0, 0, 1, 'high')
V0_star = value_function(s0, theta_trained)
print(f"\n⭐ Estimated V_0* ≈ {V0_star:.2f}")

print_greedy_trajectory(theta_trained, s0)


[Epoch 0] Average Loss = 436.668712
[Epoch 10] Average Loss = 60.486871
[Epoch 20] Average Loss = 21.072718
[Epoch 30] Average Loss = 7.370333
[Epoch 40] Average Loss = 5.255271
[Epoch 50] Average Loss = 2.854524
[Epoch 60] Average Loss = 2.259902
[Epoch 70] Average Loss = 1.635835
[Epoch 80] Average Loss = 1.457649
[Epoch 90] Average Loss = 1.192220
[Epoch 100] Average Loss = 0.954811
[Epoch 110] Average Loss = 8.178792
[Epoch 120] Average Loss = 0.927566
[Epoch 130] Average Loss = 0.927510
[Epoch 140] Average Loss = 1.114268
[Epoch 150] Average Loss = 0.836523
[Epoch 160] Average Loss = 0.862949
[Epoch 170] Average Loss = 0.939340
[Epoch 180] Average Loss = 0.888608
[Epoch 190] Average Loss = 0.981704

⭐ Estimated V_0* ≈ 151.32

🧭 Greedy Policy Trajectory from state (18, 0, 0.0, 0.0, 0, 1, 'high')
Age 18: State = (18, 0, 0.0, 0.0, 0, 1, 'high'), Action = ('delay', np.float64(-5714.29))
Age 19: State = (19, 0, -6000.0, 0.0, 0, 1, 'high'), Action = ('delay', np.float64(-11428.57))
Age 